In [ ]:
import os
import sys
import tomllib
from datetime import date
from pathlib import Path

from pyspark.sql import functions as F

os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

# Load local path configuration.
# Copy config/config.example.toml → config/config.toml and adjust paths if needed.
_cfg_file = Path("../config/config.toml")
_project_root = _cfg_file.parent.parent

with _cfg_file.open("rb") as _f:
    _cfg = tomllib.load(_f)

_src_path = (_project_root / _cfg["paths"]["src_path"]).resolve()
_data_path = (_project_root / _cfg["paths"]["data_path"]).resolve()

sys.path.insert(0, str(_src_path))

from config.settings import ETLConfig
from jobs.daily_load import run_daily_load
from jobs.initial_load import run_initial_load
from utils.spark import get_spark

spark = get_spark()
spark.sparkContext.setLogLevel("WARN")
config = ETLConfig(data_path=_data_path)

In [ ]:
initial = run_initial_load(spark, config)
print("Initial load OK")

DATE_DEBUT = date(2026, 4, 29)
DATE_FIN = date(2026, 5, 7)
daily = run_daily_load(spark, config, DATE_DEBUT, DATE_FIN, initial)
print("Daily load OK")

In [ ]:
print(f"DIM_DATE          : {initial.dim_date.count()} lignes")
print(f"  {initial.dim_date.dtypes}")
print(f"DIM_TRANSPORT_TYPE: {initial.dim_transport_type.count()} lignes")
print(f"  {initial.dim_transport_type.dtypes}")
print(f"DIM_EQUIPMENT     : {initial.dim_equipment.count()} lignes")
print(f"  {initial.dim_equipment.dtypes}")
print(f"DIM_CITY (initial): {initial.dim_city.count()} villes")
print(f"  {initial.dim_city.dtypes}")
print(f"DIM_STAFF         : {initial.dim_staff.count()} employés")
print(f"  {initial.dim_staff.dtypes}")